In [13]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
from pathlib import Path
import getpass
import os
import subprocess
import zipfile

import polars as pl
from tqdm import tqdm

PROJECT_PATH = Path("/content/drive/MyDrive/multimodal-fashion-recsys")
RAW_PATH = PROJECT_PATH / "data" / "raw"
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"
EMBEDDINGS_PATH = PROJECT_PATH / "embeddings"

ZIP_PATH = RAW_PATH / "h-and-m-personalized-fashion-recommendations.zip"

RAW_PATH.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
EMBEDDINGS_PATH.mkdir(parents=True, exist_ok=True)

print("Competition ZIP:", ZIP_PATH)
print("Exists:", ZIP_PATH.exists())

Competition ZIP: /content/drive/MyDrive/multimodal-fashion-recsys/data/raw/h-and-m-personalized-fashion-recommendations.zip
Exists: True


In [15]:
if not ZIP_PATH.exists():
    os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("Kaggle API token: ")

    subprocess.run([
        "kaggle",
        "competitions",
        "download",
        "-c",
        "h-and-m-personalized-fashion-recommendations",
        "-p",
        str(RAW_PATH)
    ], check=True)

In [16]:
print("Exists:", ZIP_PATH.exists())

if ZIP_PATH.exists():
    print(f"Size: {ZIP_PATH.stat().st_size / 1024**3:.2f} GB")

Exists: True
Size: 28.69 GB


In [17]:
articles = pl.read_csv(RAW_PATH / "articles.csv")
article_mapping = pl.read_parquet(
    PROCESSED_PATH / "article_mapping.parquet"
)

articles = (
    articles
    .select("article_id")
    .join(
        article_mapping,
        on="article_id"
    )
    .sort("article_idx")
)

print("Articles:", articles.height)
articles.head()

Articles: 105542


article_id,article_idx
i64,u32
108775015,1
108775044,2
108775051,3
110065001,4
110065002,5


In [18]:
with zipfile.ZipFile(ZIP_PATH) as archive:
    image_members = {
        name
        for name in archive.namelist()
        if name.startswith("images/")
        and name.endswith(".jpg")
    }

print("Images in ZIP:", len(image_members))

Images in ZIP: 105100


In [19]:
article_ids = articles["article_id"].to_list()
article_indices = articles["article_idx"].to_list()

rows = []

for article_idx, article_id in tqdm(
    zip(article_indices, article_ids),
    total=len(article_ids),
    desc="Building image index"
):
    article_id_str = str(article_id).zfill(10)

    image_path = (
        f"images/"
        f"{article_id_str[:3]}/"
        f"{article_id_str}.jpg"
    )

    rows.append({
        "article_idx": article_idx,
        "article_id": article_id,
        "image_path": image_path,
        "has_image": image_path in image_members
    })

image_index = pl.DataFrame(rows)

image_index.head()

Building image index: 100%|██████████| 105542/105542 [00:00<00:00, 175526.44it/s]


article_idx,article_id,image_path,has_image
i64,i64,str,bool
1,108775015,"""images/010/0108775015.jpg""",true
2,108775044,"""images/010/0108775044.jpg""",true
3,108775051,"""images/010/0108775051.jpg""",true
4,110065001,"""images/011/0110065001.jpg""",true
5,110065002,"""images/011/0110065002.jpg""",true


In [20]:
image_stats = image_index.select(
    pl.len().alias("articles"),
    pl.col("has_image").sum().alias("with_image"),
    (~pl.col("has_image")).sum().alias("without_image")
)

image_stats

articles,with_image,without_image
u32,u32,u32
105542,105100,442


In [21]:
train_items = (
    pl.scan_parquet(
        PROCESSED_PATH / "train.parquet"
    )
    .select("article_idx")
    .unique()
    .collect()
)

image_index = image_index.with_columns(
    pl.col("article_idx")
    .is_in(train_items["article_idx"])
    .alias("seen_in_train")
)

cold_image_stats = (
    image_index
    .filter(~pl.col("seen_in_train"))
    .select(
        pl.len().alias("cold_items"),
        pl.col("has_image").sum().alias("with_image"),
        (~pl.col("has_image")).sum().alias("without_image")
    )
)

cold_image_stats

/tmp/ipykernel_1065/3785421683.py:10: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  image_index = image_index.with_columns(


cold_items,with_image,without_image
u32,u32,u32
2575,2574,1


In [22]:
IMAGE_INDEX_PATH = (
    PROCESSED_PATH
    / "image_index.parquet"
)

image_index.write_parquet(
    IMAGE_INDEX_PATH,
    compression="zstd"
)

print("Saved:", IMAGE_INDEX_PATH)
print(f"Size: {IMAGE_INDEX_PATH.stat().st_size / 1024**2:.2f} MB")

Saved: /content/drive/MyDrive/multimodal-fashion-recsys/data/processed/image_index.parquet
Size: 0.60 MB


In [23]:
image_index.select(
    "article_idx",
    "article_id",
    "image_path",
    "has_image",
    "seen_in_train"
).head(10)

article_idx,article_id,image_path,has_image,seen_in_train
i64,i64,str,bool,bool
1,108775015,"""images/010/0108775015.jpg""",true,true
2,108775044,"""images/010/0108775044.jpg""",true,true
3,108775051,"""images/010/0108775051.jpg""",true,true
4,110065001,"""images/011/0110065001.jpg""",true,true
5,110065002,"""images/011/0110065002.jpg""",true,true
6,110065011,"""images/011/0110065011.jpg""",true,true
7,111565001,"""images/011/0111565001.jpg""",true,true
8,111565003,"""images/011/0111565003.jpg""",true,true
9,111586001,"""images/011/0111586001.jpg""",true,true


In [24]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("transformers") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "transformers"],
        check=True
    )

import io
import numpy as np
import torch
import torch.nn.functional as F

from PIL import Image
from transformers import CLIPImageProcessor, CLIPVisionModelWithProjection

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [25]:
MODEL_NAME = "openai/clip-vit-base-patch32"

processor = CLIPImageProcessor.from_pretrained(
    MODEL_NAME
)

vision_model = CLIPVisionModelWithProjection.from_pretrained(
    MODEL_NAME
).to(DEVICE)

vision_model.eval()

print("Model:", MODEL_NAME)
print(
    "Embedding size:",
    vision_model.config.projection_dim
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

Model: openai/clip-vit-base-patch32
Embedding size: 512


In [26]:
VISUAL_PATH = EMBEDDINGS_PATH / "clip_vit_b32"
VISUAL_PATH.mkdir(
    parents=True,
    exist_ok=True
)

BATCH_SIZE = 128
CHUNK_SIZE = 5000

visual_items = (
    image_index
    .filter(pl.col("has_image"))
    .select(
        "article_idx",
        "image_path"
    )
    .sort("article_idx")
)

print("Images to encode:", visual_items.height)
print("Chunks:", (visual_items.height + CHUNK_SIZE - 1) // CHUNK_SIZE)

Images to encode: 105100
Chunks: 22


In [27]:
def encode_image_batch(archive, batch):
    article_indices = []
    images = []

    for article_idx, image_path in batch:
        with archive.open(image_path) as file:
            image = Image.open(
                io.BytesIO(file.read())
            ).convert("RGB")

        article_indices.append(article_idx)
        images.append(image)

    pixel_values = processor(
        images=images,
        return_tensors="pt"
    )["pixel_values"].to(DEVICE)

    with torch.inference_mode():
        with torch.amp.autocast(
            "cuda",
            dtype=torch.float16
        ):
            embeddings = vision_model(
                pixel_values=pixel_values
            ).image_embeds

            embeddings = F.normalize(
                embeddings,
                p=2,
                dim=1
            )

    return (
        np.asarray(
            article_indices,
            dtype=np.int32
        ),
        embeddings.cpu().numpy().astype(
            np.float16
        )
    )

In [29]:
test_batch = visual_items.head(
    BATCH_SIZE
).rows()

with zipfile.ZipFile(ZIP_PATH) as archive:
    test_indices, test_embeddings = encode_image_batch(
        archive,
        test_batch
    )

print("Indices:", test_indices.shape)
print("Embeddings:", test_embeddings.shape)
print("Dtype:", test_embeddings.dtype)
print(
    "Mean norm:",
    np.linalg.norm(
        test_embeddings.astype(np.float32),
        axis=1
    ).mean()
)

Indices: (128,)
Embeddings: (128, 512)
Dtype: float16
Mean norm: 0.99998724


In [30]:
visual_rows = visual_items.rows()

num_chunks = (
    len(visual_rows)
    + CHUNK_SIZE
    - 1
) // CHUNK_SIZE

with zipfile.ZipFile(ZIP_PATH) as archive:
    for chunk_idx in range(num_chunks):
        chunk_path = (
            VISUAL_PATH
            / f"chunk_{chunk_idx:03d}.npz"
        )

        if chunk_path.exists():
            print(
                f"Chunk {chunk_idx + 1}/{num_chunks} already exists"
            )
            continue

        start = chunk_idx * CHUNK_SIZE
        end = min(
            start + CHUNK_SIZE,
            len(visual_rows)
        )

        chunk_rows = visual_rows[start:end]

        chunk_indices = []
        chunk_embeddings = []

        for batch_start in tqdm(
            range(0, len(chunk_rows), BATCH_SIZE),
            desc=f"Chunk {chunk_idx + 1}/{num_chunks}"
        ):
            batch = chunk_rows[
                batch_start:
                batch_start + BATCH_SIZE
            ]

            indices, embeddings = encode_image_batch(
                archive,
                batch
            )

            chunk_indices.append(indices)
            chunk_embeddings.append(embeddings)

        chunk_indices = np.concatenate(
            chunk_indices
        )

        chunk_embeddings = np.concatenate(
            chunk_embeddings
        )

        np.savez_compressed(
            chunk_path,
            article_idx=chunk_indices,
            embeddings=chunk_embeddings
        )

        print(
            f"Saved {chunk_path.name}:",
            chunk_embeddings.shape
        )

Chunk 1/22: 100%|██████████| 40/40 [02:05<00:00,  3.14s/it]


Saved chunk_000.npz: (5000, 512)


Chunk 2/22: 100%|██████████| 40/40 [02:14<00:00,  3.37s/it]


Saved chunk_001.npz: (5000, 512)


Chunk 3/22: 100%|██████████| 40/40 [02:18<00:00,  3.45s/it]


...

Saved chunk_019.npz: (5000, 512)


Chunk 21/22: 100%|██████████| 40/40 [01:43<00:00,  2.59s/it]


Saved chunk_020.npz: (5000, 512)


Chunk 22/22: 100%|██████████| 1/1 [00:02<00:00,  2.35s/it]

Saved chunk_021.npz: (100, 512)


In [31]:
VISUAL_EMBEDDINGS_PATH = (
    EMBEDDINGS_PATH
    / "clip_vit_b32_embeddings.npy"
)

visual_embeddings = np.zeros(
    (article_mapping.height + 1, 512),
    dtype=np.float16
)

chunk_paths = sorted(
    VISUAL_PATH.glob("chunk_*.npz")
)

for chunk_path in tqdm(
    chunk_paths,
    desc="Merging chunks"
):
    chunk = np.load(chunk_path)

    visual_embeddings[
        chunk["article_idx"]
    ] = chunk["embeddings"]

np.save(
    VISUAL_EMBEDDINGS_PATH,
    visual_embeddings
)

print("Shape:", visual_embeddings.shape)
print(
    f"Size: {VISUAL_EMBEDDINGS_PATH.stat().st_size / 1024**2:.2f} MB"
)

Merging chunks: 100%|██████████| 22/22 [00:09<00:00,  2.38it/s]


Shape: (105543, 512)
Size: 103.07 MB


In [32]:
embedding_norms = np.linalg.norm(
    visual_embeddings.astype(np.float32),
    axis=1
)

print(
    "Items with embeddings:",
    (embedding_norms > 0).sum()
)

print(
    "Mean norm:",
    embedding_norms[embedding_norms > 0].mean()
)

print(
    "Missing embeddings:",
    (embedding_norms == 0).sum() - 1
)

Items with embeddings: 105100
Mean norm: 1.0000001
Missing embeddings: 442


In [33]:
from datetime import date, timedelta

VAL_START = date(2020, 9, 9)

K = 12
TOP_N = 100
HISTORY_DAYS = 56
PROFILE_MAX_ITEMS = 50

train = pl.scan_parquet(PROCESSED_PATH / "train.parquet")
validation_ground_truth = pl.read_parquet(
    PROCESSED_PATH / "validation_ground_truth.parquet"
)

validation_users = validation_ground_truth.select("customer_idx")

recent_history = (
    train
    .filter(pl.col("t_dat") >= VAL_START - timedelta(days=HISTORY_DAYS))
    .join(validation_users.lazy(), on="customer_idx", how="semi")
    .sort(["customer_idx", "t_dat"])
    .group_by("customer_idx", maintain_order=True)
    .agg(
        pl.col("article_idx")
        .tail(PROFILE_MAX_ITEMS)
        .alias("history")
    )
    .collect()
)

validation_data = validation_ground_truth.join(
    recent_history,
    on="customer_idx",
    how="left"
)

print("Validation users:", validation_data.height)
print("Users with recent history:", validation_data["history"].is_not_null().sum())

Validation users: 72019
Users with recent history: 44788


In [34]:
recent_top100 = (
    train
    .filter(pl.col("t_dat") >= VAL_START - timedelta(days=14))
    .group_by("article_idx")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(TOP_N)
    .collect()["article_idx"]
    .to_list()
)

print("Fallback items:", len(recent_top100))
print(recent_top100[:12])

Fallback items: 100
[103794, 67523, 67544, 101719, 53893, 103797, 104046, 94675, 105147, 103795, 103796, 101368]


In [35]:
histories = validation_data["history"].to_list()

user_profiles = np.zeros(
    (len(histories), visual_embeddings.shape[1]),
    dtype=np.float32
)

has_visual_profile = np.zeros(
    len(histories),
    dtype=bool
)

for i, history in enumerate(tqdm(histories, desc="Building visual profiles")):
    if history is None:
        continue

    item_embeddings = visual_embeddings[
        np.asarray(history, dtype=np.int64)
    ].astype(np.float32)

    valid = np.linalg.norm(
        item_embeddings,
        axis=1
    ) > 0

    if not valid.any():
        continue

    profile = item_embeddings[valid].mean(axis=0)
    norm = np.linalg.norm(profile)

    if norm > 0:
        user_profiles[i] = profile / norm
        has_visual_profile[i] = True

print("Users with visual profile:", has_visual_profile.sum())
print("Without visual profile:", (~has_visual_profile).sum())

Building visual profiles: 100%|██████████| 72019/72019 [00:03<00:00, 19540.50it/s]

Users with visual profile: 44780
Without visual profile: 27239


In [36]:
profile_norms = np.linalg.norm(
    user_profiles[has_visual_profile],
    axis=1
)

print("Mean profile norm:", profile_norms.mean())
print("Min profile norm:", profile_norms.min())
print("Max profile norm:", profile_norms.max())

assert np.allclose(profile_norms.mean(), 1.0, atol=1e-3)

print("Tests passed")

Mean profile norm: 1.0
Min profile norm: 0.9999998
Max profile norm: 1.0000001
Tests passed


In [37]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

item_embeddings_gpu = torch.from_numpy(
    visual_embeddings
).to(
    DEVICE,
    dtype=torch.float16
)

valid_items_gpu = torch.from_numpy(
    embedding_norms > 0
).to(DEVICE)

visual_top100 = np.tile(
    np.asarray(recent_top100, dtype=np.int32),
    (len(validation_data), 1)
)

profile_indices = np.flatnonzero(
    has_visual_profile
)

BATCH_SIZE_SCORE = 256

for start in tqdm(
    range(0, len(profile_indices), BATCH_SIZE_SCORE),
    desc="Visual retrieval"
):
    indices = profile_indices[
        start:
        start + BATCH_SIZE_SCORE
    ]

    profiles = torch.from_numpy(
        user_profiles[indices]
    ).to(
        DEVICE,
        dtype=torch.float16
    )

    scores = profiles @ item_embeddings_gpu.T

    scores[:, ~valid_items_gpu] = -torch.inf

    top_items = torch.topk(
        scores,
        TOP_N,
        dim=1
    ).indices

    visual_top100[indices] = (
        top_items
        .cpu()
        .numpy()
        .astype(np.int32)
    )

print("Recommendations:", visual_top100.shape)

Visual retrieval: 100%|██████████| 175/175 [00:01<00:00, 134.68it/s]

Recommendations: (72019, 100)


In [38]:
print("First user:")
print("Actual:", validation_data["actual"][0])
print("Visual Top-12:", visual_top100[0, :12].tolist())

assert visual_top100.shape == (
    validation_data.height,
    TOP_N
)

assert visual_top100.min() > 0

print("Tests passed")

First user:
Actual: shape: (1,)
Series: '' [u32]
[
	101518
]
Visual Top-12: [74332, 13747, 97494, 63306, 13744, 66280, 54421, 86840, 94260, 75365, 51106, 73206]
Tests passed


In [39]:
def average_precision_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    hits = 0
    score = 0.0

    for i, item in enumerate(predicted[:k]):
        if item in actual:
            hits += 1
            score += hits / (i + 1)

    return score / min(len(actual), k)


def recall_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    return (
        len(actual.intersection(predicted[:k]))
        / min(len(actual), k)
    )


def ndcg_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    dcg = sum(
        1 / np.log2(i + 2)
        for i, item in enumerate(predicted[:k])
        if item in actual
    )

    ideal_hits = min(len(actual), k)

    idcg = sum(
        1 / np.log2(i + 2)
        for i in range(ideal_hits)
    )

    return dcg / idcg

In [40]:
actuals = validation_data["actual"].to_list()
visual_top12 = visual_top100[:, :K].tolist()

visual_metrics = {
    "MAP@12": sum(
        average_precision_at_k(a, p, K)
        for a, p in zip(actuals, visual_top12)
    ) / len(actuals),

    "Recall@12": sum(
        recall_at_k(a, p, K)
        for a, p in zip(actuals, visual_top12)
    ) / len(actuals),

    "NDCG@12": sum(
        ndcg_at_k(a, p, K)
        for a, p in zip(actuals, visual_top12)
    ) / len(actuals),

    "Coverage": len(set(
        visual_top100[:, :K].ravel()
    )) / (visual_embeddings.shape[0] - 1)
}

visual_metrics

{'MAP@12': 0.00862198820682814,
 'Recall@12': 0.01772097849278871,
 'NDCG@12': np.float64(0.012522619948076094),
 'Coverage': 0.3999829451782229}

In [41]:
train_item_ids = (
    train
    .select("article_idx")
    .unique()
    .collect()["article_idx"]
    .to_list()
)

train_item_set = set(train_item_ids)

cold_actuals = [
    [
        item
        for item in actual
        if item not in train_item_set
    ]
    for actual in actuals
]

cold_user_indices = [
    i
    for i, actual in enumerate(cold_actuals)
    if actual
]

cold_targets = [
    cold_actuals[i]
    for i in cold_user_indices
]

cold_predictions_12 = [
    visual_top100[i, :12].tolist()
    for i in cold_user_indices
]

cold_predictions_100 = [
    visual_top100[i].tolist()
    for i in cold_user_indices
]

cold_visual_metrics = {
    "Cold users": len(cold_user_indices),

    "Cold MAP@12": sum(
        average_precision_at_k(a, p, 12)
        for a, p in zip(
            cold_targets,
            cold_predictions_12
        )
    ) / len(cold_targets),

    "Cold Recall@12": sum(
        recall_at_k(a, p, 12)
        for a, p in zip(
            cold_targets,
            cold_predictions_12
        )
    ) / len(cold_targets),

    "Cold Recall@100": sum(
        recall_at_k(a, p, 100)
        for a, p in zip(
            cold_targets,
            cold_predictions_100
        )
    ) / len(cold_targets)
}

cold_visual_metrics

{'Cold users': 8418,
 'Cold MAP@12': 0.00036737568811695683,
 'Cold Recall@12': 0.0016008779372998902,
 'Cold Recall@100': 0.009230503795721186}

In [42]:
VISUAL_TOP100_PATH = (
    EMBEDDINGS_PATH
    / "visual_validation_top100.npz"
)

np.savez_compressed(
    VISUAL_TOP100_PATH,
    customer_idx=validation_data["customer_idx"].to_numpy(),
    recommendations=visual_top100
)

print("Saved:", VISUAL_TOP100_PATH)
print(
    f"Size: {VISUAL_TOP100_PATH.stat().st_size / 1024**2:.2f} MB"
)

Saved: /content/drive/MyDrive/multimodal-fashion-recsys/embeddings/visual_validation_top100.npz
Size: 11.45 MB


In [43]:
visual_history = (
    train
    .filter(pl.col("t_dat") >= VAL_START - timedelta(days=HISTORY_DAYS))
    .join(validation_users.lazy(), on="customer_idx", how="semi")
    .sort(["customer_idx", "t_dat"])
    .group_by("customer_idx", maintain_order=True)
    .agg(
        pl.col("article_idx").tail(PROFILE_MAX_ITEMS).alias("history"),
        pl.col("t_dat").tail(PROFILE_MAX_ITEMS).alias("dates")
    )
    .collect()
)

visual_validation_data = validation_ground_truth.join(
    visual_history,
    on="customer_idx",
    how="left"
)

visual_validation_data.head()

customer_idx,actual,history,dates
u32,list[u32],list[u32],list[date]
708122,[101518],"[13747, 97494]","[2020-08-11, 2020-08-26]"
477948,"[104506, 104364, … 101690]","[101719, 69678, … 100649]","[2020-08-27, 2020-08-27, … 2020-09-02]"
1010947,"[102724, 72966]","[87376, 52329, … 91505]","[2020-07-28, 2020-07-28, … 2020-08-03]"
1057860,"[104917, 103171, … 74491]",null,null
483825,[93735],"[99776, 94740, … 91461]","[2020-07-21, 2020-07-28, … 2020-09-07]"


In [44]:
VISUAL_HALF_LIFE = 14

histories = visual_validation_data["history"].to_list()
history_dates = visual_validation_data["dates"].to_list()

weighted_profiles = np.zeros(
    (len(histories), visual_embeddings.shape[1]),
    dtype=np.float32
)

has_weighted_profile = np.zeros(
    len(histories),
    dtype=bool
)

for i, (history, dates) in enumerate(
    tqdm(
        zip(histories, history_dates),
        total=len(histories),
        desc="Building weighted profiles"
    )
):
    if history is None:
        continue

    item_embeddings = visual_embeddings[
        np.asarray(history, dtype=np.int64)
    ].astype(np.float32)

    valid = np.linalg.norm(item_embeddings, axis=1) > 0

    if not valid.any():
        continue

    item_embeddings = item_embeddings[valid]
    valid_dates = [
        d
        for d, keep in zip(dates, valid)
        if keep
    ]

    ages = np.asarray([
        (VAL_START - d).days
        for d in valid_dates
    ])

    weights = np.exp(
        -np.log(2) * ages / VISUAL_HALF_LIFE
    ).astype(np.float32)

    profile = (
        item_embeddings * weights[:, None]
    ).sum(axis=0) / weights.sum()

    norm = np.linalg.norm(profile)

    if norm > 0:
        weighted_profiles[i] = profile / norm
        has_weighted_profile[i] = True

print("Users with weighted profile:", has_weighted_profile.sum())

Building weighted profiles: 100%|██████████| 72019/72019 [00:03<00:00, 19867.85it/s]

Users with weighted profile: 44780


In [45]:
weighted_visual_top100 = np.tile(
    np.asarray(recent_top100, dtype=np.int32),
    (len(visual_validation_data), 1)
)

profile_indices = np.flatnonzero(
    has_weighted_profile
)

for start in tqdm(
    range(0, len(profile_indices), BATCH_SIZE_SCORE),
    desc="Weighted visual retrieval"
):
    indices = profile_indices[
        start:
        start + BATCH_SIZE_SCORE
    ]

    profiles = torch.from_numpy(
        weighted_profiles[indices]
    ).to(
        DEVICE,
        dtype=torch.float16
    )

    scores = profiles @ item_embeddings_gpu.T
    scores[:, ~valid_items_gpu] = -torch.inf

    top_items = torch.topk(
        scores,
        TOP_N,
        dim=1
    ).indices

    weighted_visual_top100[indices] = (
        top_items.cpu().numpy().astype(np.int32)
    )

Weighted visual retrieval: 100%|██████████| 175/175 [00:01<00:00, 159.81it/s]


In [46]:
weighted_visual_top12 = weighted_visual_top100[:, :K].tolist()

weighted_visual_metrics = {
    "MAP@12": sum(
        average_precision_at_k(a, p, K)
        for a, p in zip(actuals, weighted_visual_top12)
    ) / len(actuals),

    "Recall@12": sum(
        recall_at_k(a, p, K)
        for a, p in zip(actuals, weighted_visual_top12)
    ) / len(actuals),

    "NDCG@12": sum(
        ndcg_at_k(a, p, K)
        for a, p in zip(actuals, weighted_visual_top12)
    ) / len(actuals),

    "Coverage": len(set(
        weighted_visual_top100[:, :K].ravel()
    )) / (visual_embeddings.shape[0] - 1)
}

weighted_visual_metrics

{'MAP@12': 0.010883868351790064,
 'Recall@12': 0.0206129196666416,
 'NDCG@12': np.float64(0.01519961719933445),
 'Coverage': 0.4131909571544977}

In [47]:
weighted_cold_predictions_12 = [
    weighted_visual_top100[i, :12].tolist()
    for i in cold_user_indices
]

weighted_cold_predictions_100 = [
    weighted_visual_top100[i].tolist()
    for i in cold_user_indices
]

weighted_cold_metrics = {
    "Cold MAP@12": sum(
        average_precision_at_k(a, p, 12)
        for a, p in zip(
            cold_targets,
            weighted_cold_predictions_12
        )
    ) / len(cold_targets),

    "Cold Recall@12": sum(
        recall_at_k(a, p, 12)
        for a, p in zip(
            cold_targets,
            weighted_cold_predictions_12
        )
    ) / len(cold_targets),

    "Cold Recall@100": sum(
        recall_at_k(a, p, 100)
        for a, p in zip(
            cold_targets,
            weighted_cold_predictions_100
        )
    ) / len(cold_targets)
}

weighted_cold_metrics

{'Cold MAP@12': 0.00043285886389949116,
 'Cold Recall@12': 0.0018800416341399947,
 'Cold Recall@100': 0.010788672798651417}

In [48]:
visual_comparison = pl.DataFrame([
    {
        "model": "Mean CLIP profile",
        **visual_metrics,
        **{
            "Cold MAP@12": cold_visual_metrics["Cold MAP@12"],
            "Cold Recall@12": cold_visual_metrics["Cold Recall@12"],
            "Cold Recall@100": cold_visual_metrics["Cold Recall@100"]
        }
    },
    {
        "model": "Recency-weighted CLIP profile",
        **weighted_visual_metrics,
        **weighted_cold_metrics
    }
]).sort("MAP@12", descending=True)

visual_comparison

model,MAP@12,Recall@12,NDCG@12,Coverage,Cold MAP@12,Cold Recall@12,Cold Recall@100
str,f64,f64,f64,f64,f64,f64,f64
"""Recency-weighted CLIP profile""",0.010884,0.020613,0.0152,0.413191,0.000433,0.00188,0.010789
"""Mean CLIP profile""",0.008622,0.017721,0.012523,0.399983,0.000367,0.001601,0.009231


In [49]:
def build_weighted_profiles(half_life):
    profiles = np.zeros(
        (len(histories), visual_embeddings.shape[1]),
        dtype=np.float32
    )

    has_profile = np.zeros(
        len(histories),
        dtype=bool
    )

    for i, (history, dates) in enumerate(
        zip(histories, history_dates)
    ):
        if history is None:
            continue

        item_embeddings = visual_embeddings[
            np.asarray(history, dtype=np.int64)
        ].astype(np.float32)

        valid = np.linalg.norm(
            item_embeddings,
            axis=1
        ) > 0

        if not valid.any():
            continue

        item_embeddings = item_embeddings[valid]

        valid_dates = [
            d
            for d, keep in zip(dates, valid)
            if keep
        ]

        ages = np.asarray([
            (VAL_START - d).days
            for d in valid_dates
        ])

        weights = np.exp(
            -np.log(2) * ages / half_life
        ).astype(np.float32)

        profile = (
            item_embeddings * weights[:, None]
        ).sum(axis=0) / weights.sum()

        norm = np.linalg.norm(profile)

        if norm > 0:
            profiles[i] = profile / norm
            has_profile[i] = True

    return profiles, has_profile

In [50]:
def retrieve_visual(profiles, has_profile):
    recommendations = np.tile(
        np.asarray(recent_top100, dtype=np.int32),
        (len(profiles), 1)
    )

    profile_indices = np.flatnonzero(has_profile)

    for start in range(
        0,
        len(profile_indices),
        BATCH_SIZE_SCORE
    ):
        indices = profile_indices[
            start:
            start + BATCH_SIZE_SCORE
        ]

        batch_profiles = torch.from_numpy(
            profiles[indices]
        ).to(
            DEVICE,
            dtype=torch.float16
        )

        scores = batch_profiles @ item_embeddings_gpu.T
        scores[:, ~valid_items_gpu] = -torch.inf

        recommendations[indices] = (
            torch.topk(
                scores,
                TOP_N,
                dim=1
            )
            .indices
            .cpu()
            .numpy()
            .astype(np.int32)
        )

    return recommendations

In [51]:
half_life_results = []
half_life_recommendations = {}

for half_life in [7, 14, 28, 56]:
    profiles, has_profile = build_weighted_profiles(
        half_life
    )

    recommendations = retrieve_visual(
        profiles,
        has_profile
    )

    predictions_12 = recommendations[:, :12].tolist()

    map12 = sum(
        average_precision_at_k(a, p, 12)
        for a, p in zip(actuals, predictions_12)
    ) / len(actuals)

    cold_predictions_100 = [
        recommendations[i].tolist()
        for i in cold_user_indices
    ]

    cold_recall100 = sum(
        recall_at_k(a, p, 100)
        for a, p in zip(
            cold_targets,
            cold_predictions_100
        )
    ) / len(cold_targets)

    half_life_results.append({
        "half_life": half_life,
        "MAP@12": map12,
        "Cold Recall@100": cold_recall100
    })

    half_life_recommendations[
        half_life
    ] = recommendations

half_life_results = (
    pl.DataFrame(half_life_results)
    .sort("MAP@12", descending=True)
)

half_life_results

half_life,MAP@12,Cold Recall@100
i64,f64,f64
7,0.01225,0.010729
14,0.010884,0.010789
28,0.009789,0.009923
56,0.009171,0.009765


In [52]:
BEST_HALF_LIFE = half_life_results["half_life"][0]

best_visual_top100 = half_life_recommendations[
    BEST_HALF_LIFE
]

print("Best half-life:", BEST_HALF_LIFE)

Best half-life: 7


In [53]:
BEST_VISUAL_PATH = (
    EMBEDDINGS_PATH
    / "visual_validation_top100_weighted.npz"
)

np.savez_compressed(
    BEST_VISUAL_PATH,
    customer_idx=visual_validation_data["customer_idx"].to_numpy(),
    recommendations=best_visual_top100,
    half_life=np.asarray([BEST_HALF_LIFE])
)

print("Saved:", BEST_VISUAL_PATH)
print(
    f"Size: {BEST_VISUAL_PATH.stat().st_size / 1024**2:.2f} MB"
)

Saved: /content/drive/MyDrive/multimodal-fashion-recsys/embeddings/visual_validation_top100_weighted.npz
Size: 11.68 MB
